In [22]:
import requests
import json
import time
import os
from datetime import datetime
from collections import defaultdict

API_TOKEN = "c9bf4127542144e8a4528ce1545a85f7"
BASE_URL = 'https://api.football-data.org/v4'
HEADERS = {'X-Auth-Token': API_TOKEN}

In [2]:
# 使用确认过的 Code 'WC' 或 ID '2000'
competition_code = "WC" 

url = f"https://api.football-data.org/v4/competitions/{competition_code}/teams"
headers = {"X-Auth-Token": API_TOKEN}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    teams = [team['name'] for team in data.get('teams', [])]
    
    print(f"成功获取到 {len(teams)} 支参赛队伍:")
    for team in teams:
        print(f"- {team}")
else:
    print(f"请求失败: {response.status_code}")
    print(response.text)

成功获取到 42 支参赛队伍:
- Uruguay
- Germany
- Spain
- Paraguay
- Argentina
- Ghana
- Brazil
- Portugal
- Japan
- Mexico
- England
- United States
- South Korea
- France
- South Africa
- Algeria
- Australia
- New Zealand
- Switzerland
- Ecuador
- Croatia
- Saudi Arabia
- Tunisia
- Senegal
- Belgium
- Morocco
- Austria
- Colombia
- Egypt
- Canada
- Haiti
- Iran
- Panama
- Cape Verde Islands
- Ivory Coast
- Qatar
- Jordan
- Uzbekistan
- Netherlands
- Norway
- Scotland
- Curaçao


In [18]:
OUTPUT_DIR = "wc2026_raw_data"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 目标：FIFA World Cup ID
WC_ID = 2000
# 定义时间分段 (每段不超过 750 天)
DATE_RANGES = [
    ("2022-01-01", "2023-12-31"),
    ("2024-01-01", "2025-12-31"),
    ("2026-01-01", "2026-12-31")
]


In [ ]:
print("🔍 正在连接 API 获取可用赛事列表...")
url = f"{BASE_URL}/competitions"

try:
    resp = requests.get(url, headers=HEADERS)
    
    if resp.status_code == 429:
        print("⛔ 错误 429: 今日配额已用完！请明天再试。")
    if resp.status_code == 403:
        print("❌ 错误 403: API Key 无效或无权限访问赛事列表。")
        print(f"响应内容: {resp.text}")
    if resp.status_code != 200:
        print(f"❌ 未知错误 {resp.status_code}: {resp.text}")

    data = resp.json()
    competitions = data.get('competitions', [])
    
    print(f"\n✅ 成功获取！你的账号当前可访问 {len(competitions)} 个赛事。\n")
    print("="*80)
    print(f"{'ID':<6} | {'Code':<6} | {'Name':<40} | {'Area':<20} | {'Plan':<10}")
    print("="*80)

    target_keywords = ["World", "Cup", "Qualification", "WC", "CONMEBOL", "UEFA"]
    found_targets = []

    for comp in competitions:
        c_id = comp.get('id')
        c_code = comp.get('code')
        c_name = comp.get('name')
        c_area = comp.get('area', {}).get('name', 'N/A')
        c_plan = comp.get('plan', 'UNKNOWN') # 显示该赛事所属的套餐等级
        
        # 格式化输出
        print(f"{c_id:<6} | {c_code:<6} | {c_name:<40} | {c_area:<20} | {c_plan:<10}")
        
        # 检查是否是我们需要的
        if any(keyword.lower() in c_name.lower() or keyword.lower() in (c_code or '').lower() for keyword in target_keywords):
            found_targets.append(comp)

    print("="*80)
    
    if found_targets:
        print("\n🎯 发现潜在的目标赛事 (World Cup / Qualification):")
        for t in found_targets:
            print(f"   👉 ID: {t['id']} | Code: {t['code']} | 名称: {t['name']}")
            print(f"      提示：尝试使用这个 ID 进行下一步数据抓取。")
    else:
        print("\n⚠️ 警告：未在列表中发现包含 'World Cup' 或 'Qualification' 的赛事。")
        print("   这可能意味着免费套餐完全锁死了世界杯相关数据。")
        print("   请检查上方列表中是否有 'Premier League', 'Champions League' 等，确认 Key 是否正常。")

except Exception as e:
    print(f"❌ 程序运行出错: {e}")

🔍 正在连接 API 获取可用赛事列表...

✅ 成功获取！你的账号当前可访问 13 个赛事。

ID     | Code   | Name                                     | Area                 | Plan      
2013   | BSA    | Campeonato Brasileiro Série A            | Brazil               | TIER_ONE  
2016   | ELC    | Championship                             | England              | TIER_ONE  
2021   | PL     | Premier League                           | England              | TIER_ONE  
2001   | CL     | UEFA Champions League                    | Europe               | TIER_ONE  
2018   | EC     | European Championship                    | Europe               | TIER_ONE  
2015   | FL1    | Ligue 1                                  | France               | TIER_ONE  
2002   | BL1    | Bundesliga                               | Germany              | TIER_ONE  
2019   | SA     | Serie A                                  | Italy                | TIER_ONE  
2003   | DED    | Eredivisie                               | Netherlands          | TIER_ONE  


In [19]:
def fetch_matches_batch():
    all_matches = []
    
    print(f"🌍 开始分批拉取 FIFA World Cup (ID: {WC_ID}) 数据...")
    print(f"   共分为 {len(DATE_RANGES)} 个时间段进行请求。\n")
    
    for i, (start, end) in enumerate(DATE_RANGES):
        print(f"[{i+1}/{len(DATE_RANGES)}] 正在请求: {start} 至 {end} ...")
        
        url = f"{BASE_URL}/competitions/{WC_ID}/matches"
        params = {
            'dateFrom': start,
            'dateTo': end,
            'status': 'FINISHED,SCHEDULED,TIMED',
            'limit': 1000
        }
        
        try:
            resp = requests.get(url, headers=HEADERS, params=params)
            
            if resp.status_code == 429:
                print("⛔ 错误 429: 配额耗尽！停止运行。")
                return None
            if resp.status_code != 200:
                print(f"❌ 请求失败 {resp.status_code}: {resp.text[:100]}")
                continue # 跳过这一段，继续下一段
            
            data = resp.json()
            matches = data.get('matches', [])
            print(f"   ✅ 获取到 {len(matches)} 场比赛")
            all_matches.extend(matches)
            
            # 稍微停顿一下，避免触发频率限制
            # import time; time.sleep(1) 
            
        except Exception as e:
            print(f"   ❌ 异常: {e}")
            continue

    print(f"\n✅ 所有批次完成！共收集到 {len(all_matches)} 场唯一比赛记录。")
    
    # 去重 (以防万一某场比赛跨越了边界被重复获取，虽然 API 通常不会这样)
    unique_matches = {m['id']: m for m in all_matches}.values()
    print(f"   🔍 去重后剩余: {len(unique_matches)} 场。")
    
    return list(unique_matches)

def split_by_team(matches):
    if not matches:
        return

    print("\n🔪 正在按队伍拆分数据...")
    
    team_matches = defaultdict(list)
    team_meta = {}

    for match in matches:
        home = match.get('homeTeam', {})
        away = match.get('awayTeam', {})
        
        h_name = home.get('name')
        a_name = away.get('name')
        h_id = home.get('id')
        a_id = away.get('id')

        if h_name: 
            team_meta[h_name] = h_id
            m = match.copy()
            m['_my_venue'] = 'HOME'
            team_matches[h_name].append(m)
        if a_name: 
            team_meta[a_name] = a_id
            m = match.copy()
            m['_my_venue'] = 'AWAY'
            team_matches[a_name].append(m)

    count = 0
    for name, games in team_matches.items():
        safe_name = name.replace(' ', '_').replace('.', '').replace("'", "")
        payload = {
            "team": {"name": name, "id": team_meta[name]},
            "source": "FIFA World Cup (2022-2026)",
            "match_count": len(games),
            "matches": games
        }
        filepath = os.path.join(OUTPUT_DIR, f"team_{safe_name}.json")
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        count += 1
    
    print(f"✅ 已生成 {count} 个队伍的数据文件。")
    print(f"💾 保存位置: {os.path.abspath(OUTPUT_DIR)}")


In [23]:
if __name__ == "__main__":
    print("="*60)
    print("🚀 启动：分批抓取模式 (解决 750 天限制)")
    print("="*60)
    
    matches = fetch_matches_batch()
    
    if matches:
        split_by_team(matches)
        print("\n🎉 大功告成！所有世界杯相关数据已本地化。")
    else:
        print("\n❌ 数据采集未完成。")

🚀 启动：分批抓取模式 (解决 750 天限制)
🌍 开始分批拉取 FIFA World Cup (ID: 2000) 数据...
   共分为 3 个时间段进行请求。

[1/3] 正在请求: 2022-01-01 至 2023-12-31 ...
❌ 请求失败 403: {"message":"The resource you are looking for is restricted and apparently not within your permission
[2/3] 正在请求: 2024-01-01 至 2025-12-31 ...
   ✅ 获取到 0 场比赛
[3/3] 正在请求: 2026-01-01 至 2026-12-31 ...
   ✅ 获取到 104 场比赛

✅ 所有批次完成！共收集到 104 场唯一比赛记录。
   🔍 去重后剩余: 104 场。

🔪 正在按队伍拆分数据...
✅ 已生成 42 个队伍的数据文件。
💾 保存位置: /Users/user/worldcup_data/wc2026_raw_data

🎉 大功告成！所有世界杯相关数据已本地化。
